In [1]:
import tiktoken
import numpy as np
from pathlib import Path
import os

In [10]:


class SimpleTokenizer:
    def __init__(self):
        self.vocab = {}
        self.reverse_vocab = {}

    def train(self, texts):
        words = set()
        for text in texts:
            words.update(text.split())
        self.vocab = {
            word: idx
            for idx, word in enumerate(sorted(words))
        }
        self.reverse_vocab = {
            idx: word
            for word, idx in self.vocab.items()
        }
    def encode(self, text):
        return " ".join(
            str(self.vocab[word]) for word in text.split()
        )

    def decode(self, ids):
        return " ".join(self.reverse_vocab[i] for i in ids)


corpus = [
    "machine learning is fun",
    "learning python is fun"
]

print(corpus)


['machine learning is fun', 'learning python is fun']


In [45]:
class SimpleTokenizer:
    def __init__(self):
        self.vocab = {}
        self.reverse_vocab = {}

    def train(self, texts):
        words = set()
        for text in texts:
            words.update(text.split())
        self.vocab = {
            word : idx
            for idx, word in enumerate(sorted(words))
        }
        self.reverse_vocab = {
            value: key
            for key, value in self.vocab.items()
        }

    def encode(self, text):
        # return " ".join(self.vocab[word].astype(str) for word in text.split())
        return [self.vocab[word] for word in text.split()]

    def decode(self, ids):
        return " ".join(self.reverse_vocab[i] for i in ids)


In [46]:
tokenizer = SimpleTokenizer()
corpus = [
    "machine learning is fun",
    "learning python is fun"
]
tokenizer.train(corpus)

In [47]:
encoded_id = tokenizer.encode("machine learning is python")
encoded_id

[3, 2, 1, 4]

In [48]:
tokenizer.decode([4,2,3,1])


'python learning machine is'

## TIKTOKEN  with text-embedding-3-small


In [32]:
with open(r"../data/processed/2606.20549v1.txt","rb") as file:
    file = file.read().decode()

In [2]:
enc = tiktoken.encoding_for_model("text-embedding-3-small")

In [45]:

for word in ["He" ,"is" ,"the", "senior" ,"member", "from", "the", "organization"]:
    tokens = enc.encode(word)
    pieces = [enc.decode([t]) for t in tokens]
    print(f"the word is ---> {word}, length ---> {len(tokens)}, decoded ----> {pieces}")

the word is ---> He, length ---> 1, decoded ----> ['He']
the word is ---> is, length ---> 1, decoded ----> ['is']
the word is ---> the, length ---> 1, decoded ----> ['the']
the word is ---> senior, length ---> 2, decoded ----> ['sen', 'ior']
the word is ---> member, length ---> 1, decoded ----> ['member']
the word is ---> from, length ---> 1, decoded ----> ['from']
the word is ---> the, length ---> 1, decoded ----> ['the']
the word is ---> organization, length ---> 1, decoded ----> ['organization']


## Splitter

In [3]:
path = Path("../data/processed/2606.20549v1.txt")

In [19]:
file_path = Path().cwd().parent / "data" / "processed" / "2606.20549v1.txt"
text = file_path.read_text(encoding="utf-8")
print(text)


Generating Robot Hands from Human
Demonstrations
Sha Yi1
Nicklas Hansen1
Xueqian Bai1
Carmelo Sferrazza2
Michael T. Tolley1
Xiaolong Wang1
1University of California San Diego
2Amazon Frontier AI & Robotics
……
Training Phase
Diverse human ﬁngertip motions from daily tasks
Robot hardware 
parameters
Joint angle 
control policy
Co-
Adaptation
Deployment Phase
High DoF 
Generalist 
Hardware
Low DoF
Specialized 
Hardware
Robot
Objects
Human 
Motion
Robot 
Motion
Figure 1: We use diverse human hand motions from daily manipulation as targets for robot hand generation.
During training, robot hardware parameters and the joint-angle control policy are optimized together to match
the observed fingertip motions. We can produce either high-DoF generalist hardware for broad teleoperation or
low-DoF specialized hardware for structured task trajectories.
Abstract: Robot learning has advanced rapidly in learning control, but learn-
ing the physical body of a robot remains much more difficult because jo

In [22]:

text = """
Transformers have revolutionized natural language processing. They enable models to learn long-range dependencies more effectively than recurrent neural networks. Modern large language models are built on top of the transformer architecture.

The self-attention mechanism allows the model to weigh the importance of different tokens in the input sequence. This helps the model capture contextual information efficiently. As model sizes increase, computational and memory requirements also increase significantly.

Researchers have proposed several techniques to improve efficiency. Sparse attention mechanisms reduce the number of attention calculations. Quantization reduces memory consumption by representing weights with fewer bits. Knowledge distillation transfers information from large models to smaller models.

Retrieval-Augmented Generation combines information retrieval with language generation. Instead of relying solely on the model's parameters, external documents can be retrieved at inference time. This often improves factual accuracy and reduces hallucinations.

A typical RAG pipeline consists of document ingestion, text extraction, chunking, embedding generation, vector database indexing, retrieval, reranking, and answer generation. The chunking stage is particularly important because it determines how much context is available to the retriever and the language model.
"""

SEPERATORS = ["\n\n", "\n", ". ", " ", ""]

def _split_on_separator(text: str, separator: str) -> list[str]:
    if separator == "":
        return list(text)
    parts = text.split(separator)
    return [p+ separator for p in parts[:-1]] + [parts[-1]]


parts = text.split("\n\n")

result = [p+ "\n" for p in parts[:-1]] + [parts[-1]]

In [31]:
from pathlib import Path
from data_curator.chunking.splitter import recursive_character_split
from data_curator.chunking.tokenizer import count_tokens
file_path = Path().cwd().parent / "data" / "processed" / "2606.20549v1.txt"
text = file_path.read_text(encoding="utf-8")
chunks = recursive_character_split(text)
print(f"Total chunks: {len(chunks)}")
print(f"Token counts: {[count_tokens(c) for c in chunks[:5]]}")
print(chunks[0][:300])

Total chunks: 34
Token counts: [587, 599, 482, 696, 584]
Generating Robot Hands from Human
Demonstrations
Sha Yi1
Nicklas Hansen1
Xueqian Bai1
Carmelo Sferrazza2
Michael T. Tolley1
Xiaolong Wang1
1University of California San Diego
2Amazon Frontier AI & Robotics
……
Training Phase
Diverse human ﬁngertip motions from daily tasks
Robot hardware 
parameters
J


In [2]:
from pathlib import Path
from data_curator.chunking.splitter import recursive_character_split
from data_curator.embeddings.sentence_embedder import embed_texts
file_path = Path().cwd().parent / "data" / "processed" / "2606.20549v1.txt"
text = file_path.read_text(encoding="utf-8")
chunks = recursive_character_split(text)

vectors = embed_texts(chunks[:3])
print(f"Num vectors: {len(vectors)}")
print(f"Vector dim: {len(vectors[0])}")

C:\Users\goutam\.conda\envs\aiagent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1/1 [00:00<00:00,  7.85it/s]

Num vectors: 3
Vector dim: 384


In [8]:
len(vectors)


3

In [9]:
from data_curator.vectorstore.qdrant_client import ensure_collection, upsert_chunks, search
from data_curator.embeddings.sentence_embedder import embed_texts, embed_query
from data_curator.config import get_settings

settings = get_settings()
ensure_collection(vector_size=settings.embedding_dim)

vectors = embed_texts(chunks)  # all chunks from before, not just 3




AttributeError: 'Settings' object has no attribute 'qdrant_url'

In [ ]:
upsert_chunks(
    chunks=chunks,
    vectors=vectors,
    paper_id="2606.20549v1",
    title="Generating Robot Hands from Human Demonstrations",
    authors=["..."],
    source_pdf_path="data/raw/2606.20549v1.pdf",
)


In [ ]:
query_vec = embed_query("How does the robot hand design framework optimize hardware?")
results = search(query_vec, top_k=3)
for r in results:
    print(f"score={r['score']:.3f} | {r['chunk_text'][:150]}")